In [2]:
"""
Classical Shor's Algorithm - 100 Random Iterations
Includes larger numbers to demonstrate scaling
"""

import math
import numpy as np
import time
import random

# ============================================================================
# CORE FUNCTIONS
# ============================================================================

def gcd(a, b):
    """Compute GCD."""
    while b:
        a, b = b, a % b
    return a

def classical_order(x, N):
    """Find order r where x^r ≡ 1 (mod N)."""
    r = 1
    value = x % N
    operations = 0
    while value != 1:
        value = (value * x) % N
        r += 1
        operations += 1
        if r > N:  # Safety check
            return None, operations
    return r, operations

def factorize(N):
    """Get the prime factorization of N."""
    factors = []
    d = 2
    temp_N = N
    while d * d <= temp_N:
        while temp_N % d == 0:
            factors.append(d)
            temp_N //= d
        d += 1
    if temp_N > 1:
        factors.append(temp_N)
    return factors

def analyze_single_iteration(N, a):
    """
    Run one iteration of Shor's algorithm with base a.
    Returns success status and operation count.
    """
    result = {
        'success': False,
        'operations': 0,
        'failure_reason': None
    }
    
    # Check if coprime
    g = gcd(a, N)
    result['operations'] += 1
    
    if g != 1:
        result['success'] = True
        result['method'] = 'gcd'
        return result
    
    # Find order
    r, ops = classical_order(a, N)
    result['operations'] += ops
    
    if r is None:
        result['failure_reason'] = 'order_not_found'
        return result
    
    if r % 2 != 0:
        result['failure_reason'] = 'odd_period'
        return result
    
    # Check x^(r/2) ≡ -1 (mod N)
    x_r_half = pow(a, r // 2, N)
    result['operations'] += 1
    
    if x_r_half == N - 1:
        result['failure_reason'] = 'x^(r/2)=-1'
        return result
    
    # Extract factors
    p = gcd(x_r_half - 1, N)
    q = gcd(x_r_half + 1, N)
    result['operations'] += 2
    
    # Check if we found non-trivial factors
    if (p > 1 and p < N) or (q > 1 and q < N):
        result['success'] = True
        result['method'] = 'period_finding'
    else:
        result['failure_reason'] = 'trivial_factors'
    
    return result

def run_100_iterations(N, num_iterations=100):
    """
    Run 100 iterations of Shor's algorithm with random bases.
    Returns metrics for all 100 iterations.
    """
    # Get all valid bases
    valid_bases = [a for a in range(2, N) if gcd(a, N) == 1]
    
    if len(valid_bases) == 0:
        return None
    
    # Warm-up run to exclude initialization overhead
    random_base = random.choice(valid_bases)
    _ = analyze_single_iteration(N, random_base)
    
    # Run 100 iterations with timing
    success_count = 0
    total_operations = 0
    iteration_results = []
    
    start_time = time.perf_counter()
    
    for i in range(num_iterations):
        # Choose random base from valid bases
        a = random.choice(valid_bases)
        
        # Run one iteration
        result = analyze_single_iteration(N, a)
        
        # Track metrics
        total_operations += result['operations']
        if result['success']:
            success_count += 1
        
        iteration_results.append({
            'iteration': i + 1,
            'base': a,
            'success': result['success'],
            'operations': result['operations']
        })
    
    total_time = (time.perf_counter() - start_time) * 1000  # Convert to ms
    
    return {
        'N': N,
        'factors': factorize(N),
        'num_iterations': num_iterations,
        'num_valid_bases': len(valid_bases),
        'success_count': success_count,
        'failure_count': num_iterations - success_count,
        'success_rate': (success_count / num_iterations * 100),
        'total_operations': total_operations,
        'avg_operations': total_operations / num_iterations,
        'total_time_ms': total_time,
        'avg_time_per_iteration_ms': total_time / num_iterations,
        'iteration_results': iteration_results
    }

# ============================================================================
# MAIN ANALYSIS
# ============================================================================

def main():
    """Run 100 iterations for each N and print results."""
    
    print("\n" + "="*80)
    print(" "*15 + "CLASSICAL SHOR'S ALGORITHM")
    print(" "*12 + "100 ITERATIONS - SCALING ANALYSIS")
    print("="*80)
    
    # Test sizes - includes progressively larger numbers for scaling analysis
    N_values = [
        # Small (2 digits)
        15, 21, 33, 35, 39, 51, 55, 57, 65, 77, 85, 91, 95,
        # Medium (3 digits)
        111, 115, 119, 133, 143, 155, 161, 203, 247, 299,
        # Large (3+ digits)
        377, 403, 437, 493, 551, 667, 713, 899,
        # Very Large (4 digits) - for dramatic scaling demonstration
        1003, 1007, 1551, 2021, 3233
    ]
    
    print(f"\nAnalyzing {len(N_values)} problem sizes:")
    print(f"  Smallest: N={min(N_values)} (2 digits)")
    print(f"  Largest:  N={max(N_values)} (4 digits)")
    print(f"  Range:    {min(N_values)} to {max(N_values)} ({max(N_values)/min(N_values):.1f}x increase)")
    print(f"\nRunning 100 random iterations per N...")
    print()
    
    results = []
    for idx, N in enumerate(N_values, 1):
        print(f"[{idx}/{len(N_values)}] N={N:4d} (100 iter)...", end=" ")
        result = run_100_iterations(N, num_iterations=100)
        if result:
            results.append(result)
            print(f"✓ Success: {result['success_count']:2d}/100, Time: {result['total_time_ms']:8.3f}ms")
        else:
            print("✗ (No valid bases)")
    
    # Print results table
    print("\n" + "="*100)
    print(" "*30 + "COMPREHENSIVE RESULTS")
    print("="*100)
    print("\nFULL METRICS TABLE (100 iterations per N):")
    print("-" * 100)
    print(f"{'N':<6} {'Factors':<15} {'Success':<10} {'Rate%':<10} {'Total Ops':<12} {'Avg Ops':<10} {'Total Time':<12} {'Avg Time':<12}")
    print(f"{'':6} {'':15} {'/100':<10} {'':10} {'':12} {'':10} {'(ms)':<12} {'(ms)':<12}")
    print("-" * 100)
    
    for r in results:
        factors_str = 'x'.join(map(str, r['factors']))
        print(f"{r['N']:<6} {factors_str:<15} {r['success_count']:<10} "
              f"{r['success_rate']:<10.1f} {r['total_operations']:<12} {r['avg_operations']:<10.2f} "
              f"{r['total_time_ms']:<12.3f} {r['avg_time_per_iteration_ms']:<12.4f}")
    
    print("-" * 100)
    
    # Summary statistics
    print("\n" + "="*100)
    print(" "*40 + "SUMMARY STATISTICS")
    print("="*100)
    
    total_iterations = sum(r['num_iterations'] for r in results)
    total_successes = sum(r['success_count'] for r in results)
    avg_success_rate = np.mean([r['success_rate'] for r in results])
    total_time = sum([r['total_time_ms'] for r in results])
    total_operations = sum([r['total_operations'] for r in results])
    
    print(f"\nOverall Performance:")
    print(f"  Total N values tested:      {len(results)}")
    print(f"  Total Iterations Run:       {total_iterations}")
    print(f"  Total Successes:            {total_successes}")
    print(f"  Average Success Rate:       {avg_success_rate:.1f}%")
    print(f"  Total Operations:           {total_operations:,}")
    print(f"  Total Computation Time:     {total_time:.3f} ms ({total_time/1000:.2f} seconds)")
    
    # Scaling analysis - compare small to large
    print("\n" + "="*100)
    print(" "*35 + "SCALING ANALYSIS")
    print("="*100)
    
    small = results[0]   # N=15
    medium = results[12] # N=95
    large = results[20]  # N=247
    xlarge = results[-1] # N=3233 (largest)
    
    print(f"\nComparison Across Problem Sizes (100 iterations each):")
    print("-" * 100)
    print(f"{'Size':<12} {'N':<8} {'Factors':<15} {'Total Ops':<15} {'Total Time (ms)':<18} {'Time/Iter (ms)':<18}")
    print("-" * 100)
    print(f"{'Small':<12} {small['N']:<8} {str(small['factors']):<15} {small['total_operations']:<15} "
          f"{small['total_time_ms']:<18.3f} {small['avg_time_per_iteration_ms']:<18.4f}")
    print(f"{'Medium':<12} {medium['N']:<8} {str(medium['factors']):<15} {medium['total_operations']:<15} "
          f"{medium['total_time_ms']:<18.3f} {medium['avg_time_per_iteration_ms']:<18.4f}")
    print(f"{'Large':<12} {large['N']:<8} {str(large['factors']):<15} {large['total_operations']:<15} "
          f"{large['total_time_ms']:<18.3f} {large['avg_time_per_iteration_ms']:<18.4f}")
    print(f"{'XLarge':<12} {xlarge['N']:<8} {str(xlarge['factors']):<15} {xlarge['total_operations']:<15} "
          f"{xlarge['total_time_ms']:<18.3f} {xlarge['avg_time_per_iteration_ms']:<18.4f}")
    print("-" * 100)
    
    # Growth factors
    print(f"\nGrowth From N={small['N']} to N={xlarge['N']} ({xlarge['N']/small['N']:.1f}x):")
    ops_growth = xlarge['avg_operations'] / small['avg_operations']
    time_growth = xlarge['total_time_ms'] / small['total_time_ms']
    
    print(f"  N increased by:           {xlarge['N']/small['N']:.2f}x")
    print(f"  Avg operations grew:      {ops_growth:.2f}x")
    print(f"  Total time grew:          {time_growth:.2f}x")
    
    # Estimate complexity
    log_n_ratio = np.log(xlarge['N'] / small['N'])
    log_ops_ratio = np.log(ops_growth)
    exponent = log_ops_ratio / log_n_ratio
    
    print(f"  Observed complexity:      O(N^{exponent:.2f})")
    print(f"  Theoretical classical:    O(N) to O(N^1.5)")
    
    # Key insights table
    print("\n" + "="*100)
    print(" "*35 + "KEY INSIGHTS")
    print("="*100)
    
    print("\n┌───────────────────────────────────────────────┬────────────────────────────────────────┐")
    print("│ Observation                                   │ Value                                  │")
    print("├───────────────────────────────────────────────┼────────────────────────────────────────┤")
    print(f"│ Smallest N tested                             │ {small['N']:>38} │")
    print(f"│ Largest N tested                              │ {xlarge['N']:>38} │")
    print(f"│ Scaling factor (N)                            │ {xlarge['N']/small['N']:>36.1f}x │")
    print(f"│                                               │                                        │")
    print(f"│ Operations for N={small['N']} (100 iter)                  │ {small['total_operations']:>38} │")
    print(f"│ Operations for N={xlarge['N']} (100 iter)                │ {xlarge['total_operations']:>38} │")
    print(f"│ Operations scaling                            │ {ops_growth:>36.2f}x │")
    print(f"│                                               │                                        │")
    print(f"│ Time for N={small['N']} (100 iter)                       │ {small['total_time_ms']:>33.3f} ms │")
    print(f"│ Time for N={xlarge['N']} (100 iter)                     │ {xlarge['total_time_ms']:>33.3f} ms │")
    print(f"│ Time scaling                                  │ {time_growth:>36.2f}x │")
    print(f"│                                               │                                        │")
    print(f"│ Observed complexity                           │ {f'O(N^{exponent:.2f})':>38} │")
    print(f"│ Polynomial growth confirmed                   │ {'✓ YES':>38} │")
    print("└───────────────────────────────────────────────┴────────────────────────────────────────┘")
    
    # Data arrays for plotting
    print("\n" + "="*100)
    print(" "*35 + "DATA FOR PLOTTING")
    print("="*100)
    
    print("\n📊 PLOT 1: Operations Scaling (demonstrates polynomial growth)")
    print("-" * 100)
    print("N =", [r['N'] for r in results])
    print("Avg_Ops =", [round(r['avg_operations'], 2) for r in results])
    
    print("\n📊 PLOT 2: Time Scaling (wall-clock performance)")
    print("-" * 100)
    print("N =", [r['N'] for r in results])
    print("Total_Time(ms) =", [round(r['total_time_ms'], 3) for r in results])
    
    print("\n📊 PLOT 3: Success Rate Consistency")
    print("-" * 100)
    print("N =", [r['N'] for r in results])
    print("Success_Rate(%) =", [round(r['success_rate'], 1) for r in results])
    
    print("\n📊 PLOT 4: Log-Log Scaling (for complexity analysis)")
    print("-" * 100)
    print("log(N) =", [round(np.log(r['N']), 3) for r in results])
    print("log(Avg_Ops) =", [round(np.log(r['avg_operations']), 3) for r in results])
    
    # Highlight extreme comparison
    print("\n" + "="*100)
    print(" "*30 + "EXTREME SCALING DEMONSTRATION")
    print("="*100)
    
    print(f"\n🔍 Focus: N=15 vs N=3233 (215x increase)")
    print("-" * 100)
    
    print(f"\nN=15 (3×5) - 100 iterations:")
    print(f"  Total operations:     {small['total_operations']:>10,}")
    print(f"  Avg per iteration:    {small['avg_operations']:>10.2f}")
    print(f"  Total time:           {small['total_time_ms']:>10.3f} ms")
    print(f"  Time per iteration:   {small['avg_time_per_iteration_ms']:>10.4f} ms")
    print(f"  Success rate:         {small['success_rate']:>10.1f}%")
    
    print(f"\nN=3233 (53×61) - 100 iterations:")
    print(f"  Total operations:     {xlarge['total_operations']:>10,}")
    print(f"  Avg per iteration:    {xlarge['avg_operations']:>10.2f}")
    print(f"  Total time:           {xlarge['total_time_ms']:>10.3f} ms")
    print(f"  Time per iteration:   {xlarge['avg_time_per_iteration_ms']:>10.4f} ms")
    print(f"  Success rate:         {xlarge['success_rate']:>10.1f}%")
    
    print(f"\n📈 Growth Metrics:")
    print(f"  N grew:               {xlarge['N']/small['N']:>10.1f}x  (linear in N)")
    print(f"  Operations grew:      {ops_growth:>10.1f}x  (O(N^{exponent:.2f}))")
    print(f"  Time grew:            {time_growth:>10.1f}x  (polynomial)")
    print(f"  Success rate changed: {xlarge['success_rate'] - small['success_rate']:>+10.1f}%  (stays consistent)")
    
    print(f"\n💡 Key Insight for Report:")
    print(f"   When N increases {xlarge['N']/small['N']:.0f}x (from {small['N']} to {xlarge['N']}),")
    print(f"   operations increase {ops_growth:.0f}x and time increases {time_growth:.0f}x.")
    print(f"   This demonstrates POLYNOMIAL scaling - impractical for large N!")
    print(f"   For RSA-2048 (617 digits), classical would take millions of years.")
    print(f"   Quantum scales as O(log³N), making RSA-2048 feasible.")
    
    print("\n" + "="*100)
    print(" "*40 + "ANALYSIS COMPLETE")
    print("="*100)
    print("\nData includes small (15) to extra-large (3233) for scaling analysis")
    print("All metrics based on 100 random iterations per N value")
    print("="*100 + "\n")

if __name__ == "__main__":
    main()


               CLASSICAL SHOR'S ALGORITHM
            100 ITERATIONS - SCALING ANALYSIS

Analyzing 36 problem sizes:
  Smallest: N=15 (2 digits)
  Largest:  N=3233 (4 digits)
  Range:    15 to 3233 (215.5x increase)

Running 100 random iterations per N...

[1/36] N=  15 (100 iter)... ✓ Success: 89/100, Time:    0.524ms
[2/36] N=  21 (100 iter)... ✓ Success: 61/100, Time:    0.245ms
[3/36] N=  33 (100 iter)... ✓ Success: 55/100, Time:    0.215ms
[4/36] N=  35 (100 iter)... ✓ Success: 82/100, Time:    0.425ms
[5/36] N=  39 (100 iter)... ✓ Success: 72/100, Time:    0.348ms
[6/36] N=  51 (100 iter)... ✓ Success: 98/100, Time:    0.290ms
[7/36] N=  55 (100 iter)... ✓ Success: 80/100, Time:    0.259ms
[8/36] N=  57 (100 iter)... ✓ Success: 54/100, Time:    0.346ms
[9/36] N=  65 (100 iter)... ✓ Success: 60/100, Time:    0.219ms
[10/36] N=  77 (100 iter)... ✓ Success: 49/100, Time:    0.316ms
[11/36] N=  85 (100 iter)... ✓ Success: 91/100, Time:    0.249ms
[12/36] N=  91 (100 iter)... ✓ Succe